In [16]:
# 7-9-2026

In [17]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

In [18]:
# same architecture as training notebook, needed to load state dict correctly
class ModularEncoder(nn.Module):
    def __init__(self, atmospheric_idx, ground_idx, fire_idx, branch_embed_dim=3, dropout=0.2):
        super().__init__()
        self.atmospheric_idx = atmospheric_idx
        self.ground_idx = ground_idx
        self.fire_idx = fire_idx

        self.atmospheric_branch = self._make_branch(len(atmospheric_idx), branch_embed_dim, dropout)
        self.ground_branch = self._make_branch(len(ground_idx), branch_embed_dim, dropout)
        self.fire_branch = self._make_branch(len(fire_idx), branch_embed_dim, dropout)

    def _make_branch(self, input_dim, embed_dim, dropout): # private
        return nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(16, embed_dim)
        ) # one branch, now input-16-16-3

    def forward(self, x):
        x_atm = x[:, self.atmospheric_idx]
        x_ground = x[:, self.ground_idx]
        x_fire = x[:, self.fire_idx]

        e_atm = self.atmospheric_branch(x_atm)
        e_ground = self.ground_branch(x_ground)
        e_fire = self.fire_branch(x_fire)

        return torch.cat([e_atm, e_ground, e_fire], dim=-1)

class PredictionHead(nn.Module):
    def __init__(self, embed_dim=9):
        super().__init__()
        # input is [e_i, e_j, e_i - e_j], outputs spearman pred
        self.net = nn.Sequential(
            nn.Linear(embed_dim * 3, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        ) # 24-16-1

    def forward(self, e_i, e_j):
        diff = e_i - e_j
        x = torch.cat([e_i, e_j, diff], dim=-1)
        return self.net(x).squeeze(-1)


class TransferModel(nn.Module):
    def __init__(self, atmospheric_idx, ground_idx, fire_idx, branch_embed_dim=3):
        super().__init__()
        self.encoder = ModularEncoder(atmospheric_idx, ground_idx, fire_idx, branch_embed_dim)
        embed_dim = branch_embed_dim * 3
        self.head = PredictionHead(embed_dim)

    def forward(self, desc_i, desc_j):
        e_i = self.encoder(desc_i)
        e_j = self.encoder(desc_j)
        return self.head(e_i, e_j)

In [19]:
transfer_matrix = pd.read_csv("../transfer-matrix/rf_transfer_matrix.csv")
transfer_matrix.set_index("Unnamed: 0", inplace=True)
transfer_matrix.index.name = None
transfer_matrix.index = transfer_matrix.index.astype(int)
transfer_matrix.columns = transfer_matrix.columns.astype(int)
transfer_matrix.head()

,0,1,2,4,5,6,7,8,11,12,...,32,33,36,37,38,39,45,46,47,49
0,0.392727,0.291660,0.127238,0.058840,0.214817,0.084705,0.149056,0.182358,0.173877,0.026375,...,0.228505,0.243433,0.149609,0.275203,0.112525,-0.011557,0.116862,0.158796,0.025454,0.029928
1,0.142172,0.395760,0.030643,0.010376,0.180707,0.064432,0.056092,0.128808,0.097788,0.039590,...,0.167477,0.032806,0.029704,0.193033,0.055264,0.066550,0.108203,0.199762,0.023240,-0.037710
2,0.196600,0.214445,0.368173,0.023968,0.207323,0.140139,0.172990,0.231019,0.156741,0.057226,...,0.197402,0.223793,0.121457,0.248807,0.134485,0.053503,-0.005610,0.253965,0.031899,0.059566
4,0.235489,0.227173,0.127394,0.410382,0.215954,0.186207,0.125068,0.196603,0.165264,0.034514,...,0.178055,0.245668,0.147579,0.284360,0.217076,-0.003941,0.037643,0.306750,0.137154,0.020968
5,0.190803,0.206530,0.097887,0.087230,0.464604,0.144950,0.159361,0.204884,0.264641,0.104493,...,0.197195,0.232923,0.117228,0.292623,0.170687,-0.020694,0.115293,0.297271,0.067951,0.051770


In [20]:
descriptors = pd.read_csv("../encoder-inputs/domain_descriptions.csv")
descriptors.set_index("domain_id", inplace=True)
descriptors.index = descriptors.index.astype(int)
descriptors.head()

,vpd_mean,vpd_std,vpd_p90,ws10_mean,ws10_std,ws10_p90,t2m_max_mean,t2m_max_std,t2m_max_p90,ndvi_mean,...,swvl4_p90,gwis_ba_mean,gwis_ba_std,gwis_ba_p90,cams_frpfire_mean,cams_frpfire_std,cams_frpfire_p90,fire_sparsity,land_cover_diversity,fire_season_length
domain_id,,,,,,,,,,,,,,,,,,,,,
0,11.229313,5.895861,19.746270,2.516127,1.052111,3.968697,302.43317,4.790887,307.46002,0.641714,...,0.479085,547.38410,1932.2546,1112.03370,0.067789,0.469590,0.097816,0.081824,0.728328,6
1,6.742952,3.144529,10.522034,1.406203,0.511182,2.132592,301.80650,4.735351,305.51370,0.796118,...,0.499585,338.66165,1157.0903,705.89197,0.063071,0.392797,0.108795,0.013560,0.218936,7
2,21.599873,10.565010,36.654785,4.066474,0.951055,5.291819,303.89624,6.380806,311.61792,0.242577,...,0.257699,3360.49340,8181.0723,8625.16300,0.162555,0.925012,0.219152,0.032987,0.602900,7
3,4.211150,2.716757,7.958644,3.166607,1.505368,5.099028,286.08792,10.954839,299.41890,0.550528,...,0.411259,328.76685,739.6544,901.44635,0.041860,0.274789,0.000000,0.001419,0.365885,3
4,2.564834,3.033652,6.938553,3.300411,1.380530,5.114551,273.42078,16.251247,293.80110,0.315671,...,0.447848,669.05206,2307.7700,1315.65490,0.185203,1.571907,0.090595,0.007227,0.615066,5


In [21]:
active_ids = transfer_matrix.index.tolist()
descriptors_active = descriptors.loc[active_ids]

# same holdout split used in training, needed to refit scaler the same way
holdout_ids = [45, 46, 47, 49]
train_ids = [d for d in active_ids if d not in holdout_ids]

In [22]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
train_desc = descriptors_active.loc[train_ids]
scaler.fit(train_desc)

desc_scaled = pd.DataFrame(
    scaler.transform(descriptors_active),
    index=descriptors_active.index,
    columns=descriptors_active.columns
)

In [23]:
atmospheric_cols = ["vpd_mean", "vpd_std", "vpd_p90", "ws10_mean", "ws10_std", "ws10_p90",
                     "t2m_max_mean", "t2m_max_std", "t2m_max_p90", "tp_mean", "tp_std", "tp_p90"]

ground_cols = ["ndvi_mean", "ndvi_std", "ndvi_p90", "swvl1_mean", "swvl1_std", "swvl1_p90",
               "swvl4_mean", "swvl4_std", "swvl4_p90", "land_cover_diversity"]

fire_cols = ["fwi_mean_mean", "fwi_mean_std", "fwi_mean_p90", "gwis_ba_mean", "gwis_ba_std", "gwis_ba_p90",
             "cams_frpfire_mean", "cams_frpfire_std", "cams_frpfire_p90", "fire_sparsity", "fire_season_length"]

atmospheric_idx = [desc_scaled.columns.get_loc(c) for c in atmospheric_cols]
ground_idx = [desc_scaled.columns.get_loc(c) for c in ground_cols]
fire_idx = [desc_scaled.columns.get_loc(c) for c in fire_cols]

In [24]:
device = torch.device("cpu")
model = TransferModel(atmospheric_idx, ground_idx, fire_idx).to(device)
model.load_state_dict(torch.load("../training/transfer_model_v3.pt", map_location=device))
model.eval()  # turn off dropout so encoder output is deterministic

TransferModel(
  (encoder): ModularEncoder(
    (atmospheric_branch): Sequential(
      (0): Linear(in_features=12, out_features=32, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.2, inplace=False)
      (3): Linear(in_features=32, out_features=16, bias=True)
      (4): ReLU()
      (5): Dropout(p=0.2, inplace=False)
      (6): Linear(in_features=16, out_features=3, bias=True)
    )
    (ground_branch): Sequential(
      (0): Linear(in_features=10, out_features=32, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.2, inplace=False)
      (3): Linear(in_features=32, out_features=16, bias=True)
      (4): ReLU()
      (5): Dropout(p=0.2, inplace=False)
      (6): Linear(in_features=16, out_features=3, bias=True)
    )
    (fire_branch): Sequential(
      (0): Linear(in_features=11, out_features=32, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.2, inplace=False)
      (3): Linear(in_features=32, out_features=16, bias=True)
      (4): ReLU()
      (5): Dropout(p=0.2, inplace=F

In [25]:
features_tensor = torch.tensor(desc_scaled.values, dtype=torch.float32).to(device) # convert to tesnor

# get embeddings
with torch.no_grad():
    embeddings_np = model.encoder(features_tensor).numpy()

In [26]:
# convert embeddings to df
embedding_cols = [f"e_{i}" for i in range(9)]
df_embeddings = pd.DataFrame(
    embeddings_np, 
    index=desc_scaled.index, 
    columns=embedding_cols
)
df_embeddings.index.name = "domain_id"
df_embeddings.reset_index(inplace=True)

In [27]:
df_embeddings.head()

,domain_id,e_0,e_1,e_2,e_3,e_4,e_5,e_6,e_7,e_8
0,0,-0.153192,0.178042,-0.220264,0.007731,-0.307213,0.146843,0.196149,0.265362,-0.027542
1,1,-0.244916,0.217204,-0.265037,0.471137,-0.148067,1.016517,0.633035,0.260105,-0.106379
2,2,0.008171,0.215184,-0.153718,-0.231967,-0.165269,-0.026749,-0.137573,0.131834,1.081505
3,4,-0.112366,-0.011225,0.986331,-0.142932,-0.113171,-0.249637,0.218600,0.164212,-0.046884
4,5,-0.218327,0.145313,-0.318968,-0.022520,-0.284123,0.194040,-0.039714,0.238028,0.039042


In [28]:
df_embeddings.to_csv("domain_embeddings_v3.csv", index=False)

In [29]:
df_embeddings.shape # 34 domains x 9dim + id

(34, 10)